# Text2SQL Evaluation — Modeling Notebook

Runs the evaluation pipeline and analyzes results.

**Prerequisites:** Run `make sample-data && make etl` first.

In [ ]:
import sys, json
sys.path.insert(0, '../src')

import pandas as pd
import plotly.express as px

from evaluation.evaluator import run_evaluation
from utils.config import get_settings

settings = get_settings()
print(f'LLM provider: {settings.llm_provider}')

In [ ]:
# Run evaluation (this may take 30-60s)
results = run_evaluation()
print(json.dumps(results['metrics'], indent=2))

In [ ]:
# Per-case results
cases = results.get('cases', [])
df = pd.DataFrame(cases)
df[['case_id', 'difficulty', 'test_type', 'is_safe', 'executed_ok', 'error']].head(20)

In [ ]:
# Accuracy by difficulty
if len(df) > 0 and 'difficulty' in df.columns:
    summary = df[df['test_type'] == 'safe_query'].groupby('difficulty')['executed_ok'].agg(['sum','count']).reset_index()
    summary['accuracy'] = summary['sum'] / summary['count']
    fig = px.bar(summary, x='difficulty', y='accuracy', title='Execution Accuracy by Difficulty',
                 color_discrete_sequence=['#2d6a9f'])
    fig.update_yaxes(range=[0, 1], tickformat='.0%')
    fig.show()

In [ ]:
# Generated SQL review
if len(df) > 0:
    for _, row in df.iterrows():
        status = '✓' if row.get('executed_ok') else '✗'
        print(f"{status} [{row.get('difficulty','?')}] {row.get('case_id','?')}")
        if row.get('sql_generated'):
            print(f"   SQL: {row['sql_generated'][:80]}...")
        print()